# GRN inference

Infers transcription factor (TF) → target gene edges for two stimulus contexts, K562 + TNF-α and THP1 + LPS. Each context uses three steps:

1. **Rank TFs** (`cp.tl.grn.rank_regulators`): signed diffusion over the OmniPath signaling network from the context's upstream signal. TFs (human TFs with OmniPath transcriptional targets) absorb the mass that reaches them and are ranked by the net activating mass. OmniPath is read from a snapshot retrieved on 2025-10-21 (`../resources/omnipath_snapshot_2025-10-21/`), so the ranking does not change as the database is updated; omit `omnipath_snapshot` to use the current OmniPath instead.
2. **Score target genes** (`cp.tl.grn.score_targets`): convert the context's tradeSeq association p-values into local FDRs (fdrtool / beta-uniform mixture) over protein-coding genes.
3. **Infer edges** (`cp.tl.grn.infer`): weight each TF → target edge by the TF's priority and the target's 1 - local FDR, then rank the edges.

Target genes come from each context's tradeSeq screen fit (`04_TradeSeqFits*`), so run the K562 + TNF-α and THP1 + LPS pipelines first. Results for each context are written to `$CHRONOPACK_DATA_DIR/grn/<context>/`.


In [1]:
import os
from pathlib import Path

import chronopack as cp

HUMAN_TFS = "../resources/hs_tfs.txt"
LIGAND_RECEPTORS = "../resources/interaction_input_CellChatDB.csv"
PROTEIN_CODING = "../resources/protein_coding_genes.csv"
# OmniPath tables retrieved on 2025-10-21 (written by cp.tl.grn.save_omnipath_snapshot).
OMNIPATH_SNAPSHOT = "../resources/omnipath_snapshot_2025-10-21"
# Folder holding one sub-folder per experiment; see analyses/README.md.
DATA_DIR = Path(os.environ.get("CHRONOPACK_DATA_DIR", "../data"))
OUTPUT_DIR = DATA_DIR / "grn"

## K562 + TNF-α

TFs are ranked downstream of the TNF ligand (seeded at its receptors); target genes come from the K562 tradeSeq association test.

In [2]:
tfs_k562 = cp.tl.grn.rank_regulators(
    source="TNF",
    is_ligand=True,
    human_tf_path=HUMAN_TFS,
    ligand_receptor_path=LIGAND_RECEPTORS,
    omnipath_snapshot=OMNIPATH_SNAPSHOT,
)
tfs_k562.head(10)

,TF,abs_pos,abs_neg,support,net,score,priority_act,priority_abs
0,RELA,0.016358,0.002585,0.018943,0.013774,0.727104,0.013774,0.013774
1,NFKB1,0.015486,0.005250,0.020736,0.010237,0.493673,0.010237,0.010237
2,STAT3,0.010025,0.002885,0.012910,0.007140,0.553066,0.007140,0.007140
3,NFKB2,0.004822,0.000570,0.005392,0.004252,0.788603,0.004252,0.004252
4,E2F1,0.004225,0.000504,0.004729,0.003721,0.786801,0.003721,0.003721
5,IRF5,0.003468,0.000273,0.003741,0.003195,0.854214,0.003195,0.003195
6,JUN,0.003080,0.000419,0.003499,0.002661,0.760492,0.002661,0.002661
7,SMAD1,0.003490,0.001058,0.004548,0.002432,0.534827,0.002432,0.002432
8,IRF7,0.002555,0.000205,0.002760,0.002350,0.851661,0.002350,0.002350
9,IRF3,0.002123,0.000280,0.002403,0.001843,0.766786,0.001843,0.001843


In [3]:
targets_k562 = cp.tl.grn.score_targets(
    DATA_DIR / "k562-tnfa.hg38.rep/intmd/sc-K562-1e4_to_tradeSeq.fixed/asso_df.1trajectory.nknots4.csv",
    valid_targets=PROTEIN_CODING,
)
targets_k562.summary

,method,n,below_0p01,below_0p05,below_0p10,below_0p20,exact_one,median,q10,q90
0,lfdr_fdrtool_fndr,10202,505,745,859,985,7760,1.000000,0.206168,1.000000
1,lfdr_fdrtool_pct0,10202,505,745,859,985,7721,1.000000,0.206036,1.000000
2,lfdr_fdrtool_locfdr,10202,505,745,859,985,7760,1.000000,0.206168,1.000000
3,bh_q,10202,710,1038,1283,1730,0,1.000000,0.046442,1.000000
4,lfdr_like_bh_q,10202,710,1038,1283,1730,0,1.000000,0.046442,1.000000
5,lfdr_bum,10202,525,693,812,980,0,0.951432,0.238231,0.971885
6,lfdr_like_logp_cap10,10202,140,153,168,203,0,0.971543,0.766745,0.999656


In [4]:
grn_k562 = cp.tl.grn.infer(targets_k562, tfs_k562, valid_targets=PROTEIN_CODING, name="ChronoPack_K562")
grn_k562.edges.head(10)

,method,regulator,target,score,rank,percentile
0,ChronoPack_K562,RELA,NFKBIA,1.751516e+06,1,1.000000
1,ChronoPack_K562,RELA,CXCL3,1.751516e+06,2,1.000000
2,ChronoPack_K562,RELA,CD69,1.751516e+06,3,0.999999
3,ChronoPack_K562,RELA,CD83,1.751516e+06,4,0.999999
4,ChronoPack_K562,RELA,ARL4A,1.751516e+06,5,0.999998
5,ChronoPack_K562,RELA,ICAM1,1.751516e+06,6,0.999998
6,ChronoPack_K562,RELA,CXCL8,1.751516e+06,7,0.999997
7,ChronoPack_K562,RELA,BTG2,1.751516e+06,8,0.999997
8,ChronoPack_K562,RELA,BIRC3,1.751516e+06,9,0.999996
9,ChronoPack_K562,RELA,DDIT4,1.751516e+06,10,0.999996


In [5]:
targets_k562.write(OUTPUT_DIR / "K562", prefix="target_scores")
tfs_k562.to_csv(OUTPUT_DIR / "K562" / "tf_ranking.csv", index=False)
grn_k562.write(OUTPUT_DIR / "K562", prefix="grn")

{'edges_csv': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/K562/grn.edges.csv', 'metadata': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/K562/grn.metadata.json', 'regulators': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/K562/grn.regulators.csv', 'target_scores': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/K562/grn.target_scores.csv'}

## THP1 + LPS

TFs are ranked downstream of the TLR4 receptor; target genes come from the THP1 tradeSeq association test.

In [6]:
tfs_thp1 = cp.tl.grn.rank_regulators(
    source="TLR4",
    human_tf_path=HUMAN_TFS,
    ligand_receptor_path=LIGAND_RECEPTORS,
    omnipath_snapshot=OMNIPATH_SNAPSHOT,
)
tfs_thp1.head(10)

,TF,abs_pos,abs_neg,support,net,score,priority_act,priority_abs
0,NFKB1,0.025628,0.003860,0.029488,0.021768,0.738185,0.021768,0.021768
1,RELA,0.025501,0.003972,0.029473,0.021530,0.730480,0.021530,0.021530
2,IRF7,0.016211,0.000504,0.016715,0.015707,0.939704,0.015707,0.015707
3,IRF5,0.006967,0.000330,0.007297,0.006637,0.909528,0.006637,0.006637
4,IRF3,0.006707,0.000317,0.007024,0.006389,0.909730,0.006389,0.006389
5,REL,0.006277,0.000181,0.006458,0.006096,0.943872,0.006096,0.006096
6,ESR1,0.005465,0.000314,0.005779,0.005151,0.891387,0.005151,0.005151
7,JUN,0.005692,0.000586,0.006278,0.005106,0.813267,0.005106,0.005106
8,STAT1,0.005242,0.000377,0.005619,0.004866,0.865933,0.004866,0.004866
9,TP53,0.007493,0.003038,0.010531,0.004455,0.423074,0.004455,0.004455


In [7]:
targets_thp1 = cp.tl.grn.score_targets(
    DATA_DIR / "thp1-k562-lps.hg38.rep/intmd/sc-THP1_to_tradeSeq.fixed/asso_df.1trajectory.nknots4.csv",
    valid_targets=PROTEIN_CODING,
)
targets_thp1.summary

,method,n,below_0p01,below_0p05,below_0p10,below_0p20,exact_one,median,q10,q90
0,lfdr_fdrtool_fndr,9481,222,308,335,429,8002,1.000000,1.000000,1.000000
1,lfdr_fdrtool_pct0,9481,169,169,169,169,8209,1.000000,1.000000,1.000000
2,lfdr_fdrtool_locfdr,9481,222,308,335,429,8002,1.000000,1.000000,1.000000
3,bh_q,9481,299,430,514,659,0,1.000000,0.367072,1.000000
4,lfdr_like_bh_q,9481,299,430,514,659,0,1.000000,0.367072,1.000000
5,lfdr_bum,9481,226,274,316,366,0,0.988336,0.865208,0.992173
6,lfdr_like_logp_cap10,9481,98,103,108,124,0,0.980430,0.856516,0.999775


In [8]:
grn_thp1 = cp.tl.grn.infer(targets_thp1, tfs_thp1, valid_targets=PROTEIN_CODING, name="ChronoPack_THP1")
grn_thp1.edges.head(10)

,method,regulator,target,score,rank,percentile
0,ChronoPack_THP1,NFKB1,CXCL8,1.724847e+06,1,1.000000
1,ChronoPack_THP1,NFKB1,CCL4,1.724847e+06,2,1.000000
2,ChronoPack_THP1,NFKB1,TNFAIP2,1.724847e+06,3,0.999999
3,ChronoPack_THP1,NFKB1,CCL4L2,1.724847e+06,4,0.999999
4,ChronoPack_THP1,NFKB1,NFKBIA,1.724847e+06,5,0.999998
5,ChronoPack_THP1,NFKB1,SOD2,1.724847e+06,6,0.999998
6,ChronoPack_THP1,NFKB1,CCL3,1.724847e+06,7,0.999997
7,ChronoPack_THP1,NFKB1,WTAP,1.724847e+06,8,0.999997
8,ChronoPack_THP1,NFKB1,CXCL3,1.724847e+06,9,0.999996
9,ChronoPack_THP1,NFKB1,TNFAIP3,1.724847e+06,10,0.999996


In [9]:
targets_thp1.write(OUTPUT_DIR / "THP1", prefix="target_scores")
tfs_thp1.to_csv(OUTPUT_DIR / "THP1" / "tf_ranking.csv", index=False)
grn_thp1.write(OUTPUT_DIR / "THP1", prefix="grn")

{'edges_csv': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/THP1/grn.edges.csv', 'metadata': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/THP1/grn.metadata.json', 'regulators': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/THP1/grn.regulators.csv', 'target_scores': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/exact_adoption_20261019/data/grn/THP1/grn.target_scores.csv'}